In [15]:
import pandas as pd

df_basin = pd.read_csv("../../CSV/compare_region_basin/CL_GL_FRS_inbasin.csv")    
df_raw   = pd.read_csv("../../CSV/compare_region_basin/country_area_timeseries.csv") 
df_area = pd.read_csv("../../CSV/gridarea/Basin_area.csv") 

In [16]:
df_area["region"] = df_area["country"] + "_" + df_area["basin"]
df_area = df_area.rename(columns={"Value": "area"})
df_area = df_area[["region", "area"]]

In [17]:
# ---------- 2. 处理 CSV②：只保留 region 模式 ----------
df_region = df_raw[df_raw["variable"].str.startswith("region_")].copy()

# 拆分项目名
df_region["type_raw"] = df_region["variable"].str.replace("region_", "", regex=False)

# 项目名称统一映射
type_map = {
    "forest": "FRS",
    "grassland": "GL",
    "agri": "CL"
}
df_region["type"] = df_region["type_raw"].map(type_map)

# 重命名列，统一结构
df_region = df_region.rename(columns={
    "basin": "region",
    "area": "region_value"
})

df_region = df_region[["region", "year", "type", "region_value"]]

# ---------- 3. 处理 CSV①（basin 模式） ----------
df_basin = df_basin.rename(columns={
    "value": "basin_value"
})
df_basin = df_basin[["region", "year", "type", "basin_value"]]

# ---------- 4. 合并并计算差值 ----------
df_diff = pd.merge(
    df_region,
    df_basin,
    on=["region", "year", "type"],
    how="inner"
)

df_diff["diff_region-basin"] = (
    df_diff["region_value"] - df_diff["basin_value"]
)

In [19]:
df_diff = df_diff.merge(
    df_area,
    on=["region"],
    how="left"
)
df_diff["diff/area"]=df_diff["diff_region-basin"]/df_diff["area"]
num_cols = df_diff.columns[2:]
df_diff[num_cols] = df_diff[num_cols].applymap(lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else x)
# ---------- 5. 输出 ----------
df_diff.to_csv("../../CSV/compare_region_basin/region_basin_diff.csv", index=False)

print("Saved: region_minus_basin_diff.csv")

Saved: region_minus_basin_diff.csv


C:\Users\ZSR\AppData\Local\Temp\ipykernel_15208\3975541650.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_diff[num_cols] = df_diff[num_cols].applymap(lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else x)
